In [4]:
from torch.optim import Adam, SGD
from torch.nn import functional as F
import torch
import polars as pl
import torch.nn as nn
import sys
import os
from pathlib import Path
sys.path.append(str(Path(os.path.abspath("..")).resolve()))
from src.utils.data import load_yaml

In [2]:
model = nn.Linear(1, 1)

In [6]:
next(model.parameters()).is_cuda

False

In [7]:
model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Linear(in_features=1, out_features=1, bias=True)

In [8]:
next(model.parameters()).is_cuda

True

### LSTMの使い方

#### cやhはどのように与えるの？
>h_0: tensor of shape (D * num_layers, H_out) for unbatched input
or (D * num_layers, N, H_out) containing the initial hidden state
for each element in the input sequence.
Defaults to zeros if (h_0, c_0) is not provided.

>c_0: tensor of shape (D * num_layers, H_cell) for unbatched input
or (D * num_layers, N, H_cell) containing the initial cell state
for each element in the input sequence.
Defaults to zeros if (h_0, c_0) is not provided.

In [47]:
MODEL_YAML = 'model.yaml'
model_config = load_yaml(f"../config/{MODEL_YAML}")
lstm_config = model_config["decoder"]["lstm"]

In [48]:
lstm_config

{'input_size': 10,
 'hidden_size': 5,
 'num_layers': 1,
 'bias': True,
 'batch_first': True,
 'dropout': 0.1}

In [49]:
T = 3
N = 10

In [56]:
model = nn.LSTM(**lstm_config)

### LSTMの重みの説明
weight_ih_l0   # 入力 x にかかる重み  (Wx)  
weight_hh_l0   # 前時刻の隠れ状態 h にかかる重み  (Wh)  
bias_ih_l0  
bias_hh_l0  

In [57]:
for name, param in model.named_parameters():
    print(name, param.shape)

weight_ih_l0 torch.Size([20, 10])
weight_hh_l0 torch.Size([20, 5])
bias_ih_l0 torch.Size([20])
bias_hh_l0 torch.Size([20])


In [38]:
x = torch.randint(0, 11, (N, T, lstm_config["hidden_size"])).to(torch.float32)
x.shape, x.max()

(torch.Size([10, 3, 5]), tensor(10.))

In [42]:
h = torch.randint(0, 11, (1, N, lstm_config["hidden_size"])).to(torch.float32)
c = torch.randint(0, 11, (1, N, lstm_config["hidden_size"])).to(torch.float32)
h.shape, c.shape

(torch.Size([1, 10, 5]), torch.Size([1, 10, 5]))

In [43]:
output, (hn, cn) = model(x, (h, c))

RuntimeError: input.size(-1) must be equal to input_size. Expected 10, got 5